In [1]:
%pip install -q -U transformers peft accelerate bitsandbytes
%pip uninstall -y -q torchao   # newer torchao breaks the peft import on current Colab

In [2]:
import os, sys, json, re, csv, gc, time, math, random, collections, shutil
from datetime import date, datetime, timedelta
from google.colab import drive
drive.mount('/content/drive')

WORK = os.environ.get('WORK', '/content/drive/MyDrive/vima3ya')
DATA = f'{WORK}/data'; RUNS = f'{WORK}/runs'; OUT = f'{WORK}/outputs'
for d in (RUNS, OUT): os.makedirs(d, exist_ok=True)

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
SEED = 42
random.seed(SEED)

missing = [f for f in ('train.jsonl', 'dev.jsonl', 'test_inputs.jsonl', 'score.py') if not os.path.exists(f'{DATA}/{f}')]
assert not missing, f'put these files in {DATA}: {missing}'          # no dummy fallbacks: a missing scorer must stop the run
sys.path.insert(0, DATA)
from score import score, load, KEYS, same, parse                      # the official scorer, unchanged
assert len(KEYS) == 8, KEYS

def read_jsonl(p): return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]
train_raw = read_jsonl(f'{DATA}/train.jsonl')
dev       = read_jsonl(f'{DATA}/dev.jsonl')
test_in   = read_jsonl(f'{DATA}/test_inputs.jsonl')
print(len(train_raw), len(dev), len(test_in))

Mounted at /content/drive
1445 200 400


In [3]:
import json,re
from datetime import date,timedelta
MON={m:i+1 for i,m in enumerate('jan feb mar apr may jun jul aug sep oct nov dec'.split())}
MONRE=r'(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*'
def split(u):
    head,_,body=u.partition('\n\n'); rec=re.search(r'Received: (\d{4})-(\d\d)-(\d\d)',head)
    return date(*map(int,rec.groups())),body
def clean_body(b):
    lines=[l for l in b.split('\n') if not l.lstrip().startswith('>')]
    return '\n'.join(lines)
FOOT=re.compile(r'Our team will get back|Please do not share your OTP|You can track the status|If you have already received|We value your trust|Your request has been registered',re.I)
def sentences(b):
    b=re.sub(r'\b(Rs|INR)\.\s*',r'\1 ',b)
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+|\n',b) if s.strip()]
def txn_id(b):
    for m in re.finditer(r'\bn\s*p[\s\-]*((?:\d[\s\-]*){10})(?!\d)',b,re.I):
        d=re.sub(r'\D','',m.group(1))
        if len(d)==10: return 'NP'+d
    return None
def amount(b):
    b=re.sub(r'\bn\s*p[\s\-]*(?:\d[\s\-]*){10}','',b,flags=re.I)
    best=None
    for s in sentences(b):
        if FOOT.search(s) or re.search(r'balance|limit|\bfee|pade hain|abhi .{0,20}(pade|hai)\b',s,re.I): continue
        for m in re.finditer(r'(?:₹|rs\.?|inr)\s*([\d,]+(?:\.\d+)?)\s*(k\b|lakhs?\b)?|([\d,]+(?:\.\d+)?)\s*(k|lakhs?|rupees|rupaye|rupee|rs)\b',s,re.I):
            num=m.group(1) or m.group(3); suf=(m.group(2) or m.group(4) or '').lower()
            v=float(num.replace(',',''))
            if suf=='k': v*=1000
            elif suf.startswith('lakh'): v*=100000
            if best is None: best=int(round(v*100))
    return best
def channel(b):
    t=b.lower()
    if re.search(r'internet banking|netbanking|net banking',t): return 'netbanking'
    if re.search(r'\bupi\b',t): return 'upi'
    if re.search(r'(?<!scratch )(?<!srcatch )\bcard\b|visa|mastercard|\bcard se\b',t): return 'card'
    if re.search(r'\bwallet\b',t): return 'wallet'
    return None
def txn_date(rec,b):
    for s in sentences(b):
        if FOOT.search(s) or re.search(r'ticket|ref no|bank ref|version',s,re.I): continue
        sl=s.lower()
        m=re.search(r'\b(\d{1,2})[-/](\d{1,2})[-/](\d{2,4})\b',s)
        if m:
            d,mo,y=map(int,m.groups()); y+=2000 if y<100 else 0
            return date(y,mo,d).isoformat()
        m=re.search(r'\b(\d{1,2})(?:st|nd|rd|th)?\s+'+MONRE+r'(?:\s+(\d{4}))?',s,re.I)
        m2=re.search(MONRE+r'\s+(\d{1,2})(?:st|nd|rd|th)?(?:\s*,?\s*(\d{4}))?\b',s,re.I)
        for mm,order in ((m,'dm'),(m2,'md')):
            if mm:
                g=mm.groups()
                if order=='dm': d,mon,y=int(g[0]),g[1],g[2]
                else: mon,d,y=g[0],int(g[1]),g[2]
                mon=MON[mon[:3].lower()]
                if y: return date(int(y),mon,d).isoformat()
                c=date(rec.year,mon,d)
                if c>rec: c=date(rec.year-1,mon,d)
                return c.isoformat()
        if re.search(r'day before yesterday|parso',sl): return (rec-timedelta(2)).isoformat()
        if re.search(r'yesterday|\bkal\b',sl): return (rec-timedelta(1)).isoformat()
        if re.search(r'\btoday\b|\baaj\b',sl): return rec.isoformat()
        m=re.search(r'(\d+)\s*(?:days? ago|din pehle)',sl)
        if m: return (rec-timedelta(int(m.group(1)))).isoformat()
    return None
def language(b):
    strong=len(re.findall(r"\b(hai|nahi|nahin|kya|mera|meri|mere|kaise|hua|gaya|gayi|karo|karna|wapas|paise|paisa|abhi|bhai|dhanyavaad|kripya|madad|rupaye|chahiye|batao|hoon|mujhe|humein|pehle|baar|tha|thi|kat)\b",b,re.I))
    return 'hi-en' if strong>=1 else 'en'
REPEAT=re.compile(r'second complaint|again and again|pehle bhi complaint|already written \d+ times|third time|teesri baar|contacted support about this before|baar baar|bar bar|keep raising|already (contacted|complained)',re.I)
LEGAL=re.compile(r'\bRBI\b|ombudsman|consumer court|lawyer|legal notice|legal action|advocate',re.I)
def ref(row):
    u=row['messages'][1]['content']; rec,b=split(u); b=clean_body(b)
    nb='\n'.join(s for s in sentences(b) if not FOOT.search(s))
    return dict(txn_id=txn_id(nb),amount_paise=amount(nb),txn_date=txn_date(rec,nb),channel=channel(nb),language=language(nb),
                repeat=bool(REPEAT.search(nb)),legal=bool(LEGAL.search(nb)))
def prio(cat,amt,legal):
    a=(amt or 0)/100
    if cat=='fraud' or a>=50000 or legal: return 'P1'
    if (cat in('payment_failed','refund') and a>=5000) or cat=='account_access': return 'P2'
    return 'P3'

In [4]:
def check_against_ref(rows):
    bad = collections.defaultdict(list)
    for r in rows:
        g = json.loads(r['messages'][2]['content']); p = ref(r)
        txn = g['category'] not in ('account_access', 'kyc', 'other')
        for k in ('txn_id', 'amount_paise', 'txn_date', 'channel'):
            if g[k] != (p[k] if txn else None): bad[k].append(r['id'])
        if g['language'] != p['language']: bad['language'].append(r['id'])
        if g['priority'] != prio(g['category'], g['amount_paise'], p['legal']): bad['priority'].append(r['id'])
        if g['needs_human'] != (g['priority'] == 'P1' or p['repeat']): bad['needs_human'].append(r['id'])
    return bad

bad_dev = check_against_ref(dev)
print('disagreements with clean dev:', {k: len(v) for k, v in bad_dev.items()} or 'none')
assert not bad_dev, 'labeller does not reproduce dev, fix the labeller before auditing train'

disagreements with clean dev: none


In [5]:
U = lambda r: r['messages'][1]['content']
A = lambda r: r['messages'][2]['content']
body_of = lambda r: U(r).split('\n\n', 1)[-1].strip()
dev_texts = {U(r) for r in dev}

found = collections.defaultdict(list)
first_seen = {}
for r in train_raw:
    i = r['id']
    if len(body_of(r)) < 12 or re.fullmatch(r'(test( test)?( \d+)?|asdf( asdf)*)', body_of(r).lower()): found['junk text'].append(i)
    if U(r) in dev_texts: found['copy of a dev ticket (leakage)'].append(i)
    if U(r) in first_seen: found['exact duplicate'].append(i)
    first_seen.setdefault(U(r), i)
    try: g = json.loads(A(r))
    except Exception: found['truncated JSON label'].append(i); continue
    if set(g) != set(KEYS):                 found['v1 schema (prio, dd/mm/yyyy)'].append(i); continue
    p = ref(r); txn = g['category'] not in ('account_access', 'kyc', 'other')
    if g['txn_id'] and g['txn_id'] != re.sub(r'[^NP\d]', '', g['txn_id'].upper()): found['txn_id not normalised'].append(i)
    if txn and g['amount_paise'] and p['amount_paise'] and g['amount_paise'] != p['amount_paise'] and abs(p['amount_paise'] - 100 * g['amount_paise']) <= 100:
        found['amount in rupees, not paise'].append(i)
    if g['priority'] != prio(g['category'], g['amount_paise'], p['legal']): found['priority breaks SCHEMA rules'].append(i)
    if g['needs_human'] != (g['priority'] == 'P1' or p['repeat']):         found['needs_human inconsistent'].append(i)
for k, v in sorted(found.items(), key=lambda kv: -len(kv[1])):
    print(f'{k:36} {len(v):4}   e.g. {v[:3]}')
print('rows touched by at least one detector:', len(set().union(*map(set, found.values()))))

v1 schema (prio, dd/mm/yyyy)           70   e.g. ['tr-00020', 'tr-00049', 'tr-00058']
priority breaks SCHEMA rules           65   e.g. ['tr-00013', 'tr-00018', 'tr-00023']
exact duplicate                        60   e.g. ['tr-00156', 'tr-00208', 'tr-00211']
needs_human inconsistent               45   e.g. ['tr-00018', 'tr-00023', 'tr-00045']
copy of a dev ticket (leakage)         30   e.g. ['tr-00008', 'tr-00086', 'tr-00130']
truncated JSON label                   25   e.g. ['tr-00060', 'tr-00095', 'tr-00125']
amount in rupees, not paise            25   e.g. ['tr-00073', 'tr-00144', 'tr-00167']
txn_id not normalised                  20   e.g. ['tr-00071', 'tr-00147', 'tr-00151']
junk text                              10   e.g. ['tr-00034', 'tr-00164', 'tr-00318']
rows touched by at least one detector: 300


In [6]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.pipeline import make_pipeline, make_union

def text_for_cls(r):
    _, b = split(U(r)); b = clean_body(b)
    b = '\n'.join(s for s in sentences(b) if not FOOT.search(s))
    b = re.sub(r'\bn\s*p[\s\-]*(?:\d[\s\-]*){10}', ' ', b, flags=re.I); b = re.sub(r'\d+', ' ', b)
    return b.lower()
def cat_of(r):
    try: return json.loads(A(r))['category']
    except Exception:
        m = re.search(r'"category": "(\w+)"', A(r)); return m.group(1) if m else None
rows_c = [r for r in train_raw if cat_of(r) and len(text_for_cls(r).strip()) > 10]
Xc = [text_for_cls(r) for r in rows_c]; yc = np.array([cat_of(r) for r in rows_c])
mk = lambda: make_pipeline(make_union(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                                      TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=3, sublinear_tf=True)),
                           LogisticRegression(C=20, max_iter=2000))
P = cross_val_predict(mk(), Xc, yc, cv=StratifiedKFold(5, shuffle=True, random_state=0), method='predict_proba')
cls = np.array(sorted(set(yc))); pred = cls[P.argmax(1)]; conf = P.max(1)
print('5-fold CV accuracy of the category auditor: %.3f' % np.mean(pred == yc))
sus = [(rows_c[i]['id'], yc[i], pred[i], round(float(conf[i]), 2)) for i in range(len(rows_c)) if pred[i] != yc[i] and conf[i] > 0.6]
print('confident disagreements:', len(sus))
by_id = {r['id']: r for r in train_raw}
for i, l, p_, c in sus:
    print(f'  {i} label={l} auditor={p_} ({c}) | ' + re.sub(r'\s+', ' ', body_of(by_id[i]))[:150])

5-fold CV accuracy of the category auditor: 0.979
confident disagreements: 7
  tr-00257 label=fraud auditor=refund (0.76) | Subject: Help needed Good morning, I see a payment on my statement for a place I didn't make a purchase at. It was from the NimbusPay wallet. It was f
  tr-00729 label=refund auditor=payment_failed (0.88) | Hi, My card was declined but it shows as multiple charges? Rs. 2728 is the amount. It took place today.
  tr-00975 label=fraud auditor=refund (0.6) | URGENT!! I don't understand where this charge came from. 944 rupees is the amount. Date of transaction: Aug 29. Mode: debit card. Please look into it.
  tr-01128 label=other auditor=account_access (0.65) | Subject: Complaint Good morning, can i create my own pin right away. I am on the iOS app. My earlier ticket number is #92893. Please look into it. Reg
  tr-01202 label=other auditor=account_access (0.73) | [IVR transcript] Hey, I'm looking in the App and can't find my PIN, where should I look? Please look into i

In [7]:
KEYS=["category","priority","amount_paise","txn_date","txn_id","channel","language","needs_human"]
U=lambda r:r['messages'][1]['content'];A=lambda r:r['messages'][2]['content']
def body(r): return U(r).split('\n\n',1)[-1].strip()
def is_junk(r): return len(body(r))<12 or re.fullmatch(r'(test( test)?( \d+)?|asdf( asdf)*)',body(r).lower()) is not None
def parse_prefix(a):
    """Recover the complete key:value pairs from a truncated JSON string."""
    out={}
    for m in re.finditer(r'"(\w+)": ("[^"]*"|null|true|false|-?\d+)(?=, "|\})',a):
        out[m.group(1)]=json.loads(m.group(2))
    return out
def clean_dataset(train,dev,log):
    dev_texts={U(r) for r in dev};seen=set();out=[]
    for r in train:
        i=r['id'];txt=U(r)
        if is_junk(r): log.append((i,'drop','junk text ("%s") with fabricated label'%body(r)[:20]));continue
        if txt in dev_texts: log.append((i,'drop','exact copy of a dev ticket (leakage)'));continue
        if txt in seen: log.append((i,'drop','exact duplicate of an earlier train row'));continue
        seen.add(txt)
        p=ref(r);why=[]
        try: g=json.loads(A(r))
        except Exception:
            pre=parse_prefix(A(r));cat=pre.get('category')
            if cat is None: log.append((i,'drop','truncated JSON, category lost'));continue
            g=dict(category=cat);why.append('truncated JSON rebuilt from surviving prefix + rules')
            txn=cat not in('account_access','kyc','other')
            for k in('txn_id','amount_paise','txn_date','channel'): g[k]=p[k] if txn else None
            g['language']=p['language'];g['priority']=prio(cat,g['amount_paise'],p['legal'])
            g['needs_human']=bool(g['priority']=='P1' or p['repeat'])
            conflict=[k for k,v in pre.items() if k!='category' and g.get(k)!=v]
            if conflict: log.append((i,'drop','truncated JSON; rebuilt value disagrees with surviving prefix on '+','.join(conflict)));continue
            out.append((r,g,why));continue
        if 'prio' in g and 'priority' not in g:
            g['priority']=g.pop('prio');why.append('v1 schema (prio low/medium/high) converted to v2')
            if g['txn_date'] and re.fullmatch(r'\d\d/\d\d/\d{4}',g['txn_date']):
                g['txn_date']=datetime.strptime(g['txn_date'],'%d/%m/%Y').date().isoformat();why.append('dd/mm/yyyy date to ISO')
        g={k:g[k] for k in KEYS}
        txn=g['category'] not in('account_access','kyc','other')
        if g['amount_paise'] is not None and txn and p['amount_paise'] and g['amount_paise']!=p['amount_paise'] and abs(p['amount_paise']-100*g['amount_paise'])<=100:
            why.append('amount stored in rupees, converted to paise (%d to %d)'%(g['amount_paise'],p['amount_paise']));g['amount_paise']=p['amount_paise']
        if g['txn_id'] and g['txn_id']!=re.sub(r'[^NP\d]','',g['txn_id'].upper()):
            why.append('txn_id not normalised (%s)'%g['txn_id']);g['txn_id']=re.sub(r'[^NP\d]','',g['txn_id'].upper())
        pr=prio(g['category'],g['amount_paise'],p['legal'])
        if g['priority']!=pr: why.append('priority %s violates SCHEMA rules, set to %s'%(g['priority'],pr));g['priority']=pr
        nh=bool(g['priority']=='P1' or p['repeat'])
        if g['needs_human']!=nh: why.append('needs_human inconsistent with rules, set to %s'%nh);g['needs_human']=nh
        out.append((r,g,why))
    final=[]
    for r,g,why in out:
        if why: log.append((r['id'],'fix','; '.join(why)))
        final.append({'id':r['id'],'messages':r['messages'][:2]+[{'role':'assistant','content':json.dumps(g)}]})
    return final

In [8]:
log = []
train_clean = clean_dataset(train_raw, dev, log)
c = collections.Counter(a for _, a, _ in log)
print(f'kept {len(train_clean)} of {len(train_raw)} | dropped {c["drop"]} | fixed {c["fix"]}')

with open(f'{OUT}/cleaning_log.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['id', 'action', 'reason']); w.writerows(sorted(log))

ld = []; _ = clean_dataset(dev, [], ld)
print('cleaner on clean dev flags', len(ld), 'rows (expected 0)')
assert len(ld) == 0

bad_tr = check_against_ref(train_clean)
print('residual disagreements after cleaning:', {k: v for k, v in bad_tr.items()})
LEAKED_DEV_IDS = {next(d['id'] for d in dev if U(d) == U(r)) for r in train_raw if U(r) in dev_texts}
print('dev rows that also sit in raw train:', len(LEAKED_DEV_IDS))

kept 1345 of 1445 | dropped 100 | fixed 200
cleaner on clean dev flags 0 rows (expected 0)
residual disagreements after cleaning: {'amount_paise': ['tr-00171']}
dev rows that also sit in raw train: 30


In [9]:
import numpy as np, torch, torch.nn.functional as F, shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training
from score import same, parse
import transformers, peft
print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)
assert torch.cuda.is_available(), 'switch the runtime to a T4 GPU'
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GB')

torch 2.14.1+cu130 | transformers 5.22.2 | peft 0.21.2
Tesla T4 14.6 GB


In [10]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None: tok.pad_token = tok.eos_token

def render(msgs, gen_prompt):
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=gen_prompt)

def encode_example(row):
    """ids = chat-template prompt + assistant JSON + end-of-turn. labels = -100 on the prompt."""
    msgs = row['messages']
    prompt, full = render(msgs[:2], True), render(msgs, False)
    assert full.startswith(prompt)
    p = tok(prompt, add_special_tokens=False)['input_ids']
    t = tok(full[len(prompt):], add_special_tokens=False)['input_ids']
    return p + t, [-100] * len(p) + t

ex = train_clean[0]
ids, lab = encode_example(ex)
full_ids = tok(render(ex['messages'], False), add_special_tokens=False)['input_ids']
print('prompt+target ids identical to tokenising the whole conversation at once:', full_ids == ids)
print(render(ex['messages'][:2], True)[-120:].replace('\n', '\\n'))
print('LOSS ON  :', repr(tok.decode([t for t in lab if t != -100])))
print('LOSS OFF :', len([t for t in lab if t == -100]), 'prompt tokens')

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

prompt+target ids identical to tokenising the whole conversation at once: True
nt balance is 258 rupees right now. The txn id is NP1599046332. Waiting for your reply.<|im_end|>\n<|im_start|>assistant\n
LOSS ON  : '{"category": "payment_failed", "priority": "P3", "amount_paise": 470000, "txn_date": "2026-07-25", "txn_id": "NP1599046332", "channel": "netbanking", "language": "en", "needs_human": false}<|im_end|>\n'
LOSS OFF : 133 prompt tokens


In [11]:
HDR=re.compile(r'Received: \d{4}-\d\d-\d\d \(\w{3}\)')
def body_for_date(u):
    rec,b=split(u);b=clean_body(b)
    return rec,'\n'.join(s for s in sentences(b) if not FOOT.search(s))
def redate(row,new):
    """Copy of `row` with a new Received header and txn_date recomputed by the SCHEMA rules, or None."""
    u=row['messages'][1]['content'];g=json.loads(row['messages'][2]['content'])
    if g['txn_date'] is None: return None
    rec,b=body_for_date(u)
    if txn_date(rec,b)!=g['txn_date']: return None          # rule does not reproduce this label: skip
    nd=txn_date(new,b)
    if nd==g['txn_date']: return None                        # date does not depend on Received
    g2=dict(g,txn_date=nd)
    u2=HDR.sub('Received: %s (%s)'%(new.isoformat(),new.strftime('%a')),u,count=1)
    return {'id':'aug-'+row['id'],'messages':[row['messages'][0],{'role':'user','content':u2},{'role':'assistant','content':json.dumps(g2)}]}
def augment(rows,seed=0):
    rng=random.Random(seed);out=[]
    for r in rows:
        new=date(2026,6,1)+timedelta(days=rng.randrange(0,112))
        a=redate(r,new)
        if a: out.append(a)
    return out

In [12]:
aug_rows = augment(train_clean, seed=SEED)
probe_rows = []
for k in (1, 2):
    probe_rows += [a for a in (redate(r, date(2026, 6, 1) + timedelta(days=random.Random(SEED + k + j).randrange(112)))
                               for j, r in enumerate(dev)) if a]
for i, a in enumerate(probe_rows): a['id'] = f'probe-{i:04d}'
print('augmentation copies:', len(aug_rows), '| prev-year dates among them:', sum(json.loads(A(a))['txn_date'].startswith('2025') for a in aug_rows))
print('probe rows:', len(probe_rows))
probe = probe_rows

augmentation copies: 263 | prev-year dates among them: 78
probe rows: 86


In [13]:
all_rows = train_raw + aug_rows
lens = np.array([len(encode_example(r)[0]) for r in all_rows])
print('token length percentiles 50/90/95/99/100:', [int(np.percentile(lens, q)) for q in (50, 90, 95, 99, 100)])
print('rows over 512 tokens:', int((lens > 512).sum()), '| over 1024:', int((lens > 1024).sum()), 'of', len(lens))
tl = np.array([len(tok(render(r['messages'][:2], True), add_special_tokens=False)['input_ids']) for r in test_in])
print('test prompt tokens p99/max:', int(np.percentile(tl, 99)), int(tl.max()), '(+200 new tokens at inference)')
MAX_LEN = int(math.ceil(lens.max() / 64) * 64)
print('MAX_LEN =', MAX_LEN)

token length percentiles 50/90/95/99/100: [172, 214, 225, 1787, 2215]
rows over 512 tokens: 44 | over 1024: 44 of 1708
test prompt tokens p99/max: 1827 2089 (+200 new tokens at inference)
MAX_LEN = 2240


In [14]:
TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
DEFAULT_CFG = dict(quant4=False, r=16, alpha=32, dropout=0.05, lr=2e-4, epochs=3,
                   max_tokens=8192, max_bs=16, target_ex=16, warmup=0.05, seed=SEED)

def load_base(quant4=False):
    kw = dict(device_map={'': 0})
    if quant4:
        kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                                       bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    try:    return AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, **kw)
    except TypeError: return AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, **kw)

def to_eval(model):
    inner = model.get_base_model() if hasattr(model, 'get_base_model') else model
    try: inner.gradient_checkpointing_disable()
    except Exception: pass
    inner.config.use_cache = True; model.eval()

def free():
    gc.collect(); torch.cuda.empty_cache()

@torch.no_grad()
def generate(model, rows, system=None, bs=16, char_budget=12000, max_new_tokens=200, **gen_kw):
    """Greedy decoding, chat template, raw text out. Batches are capped by a character budget so long threads run alone."""
    model.eval(); tok.padding_side = 'left'
    prompts = []
    for r in rows:
        m = r['messages'][:2]
        if system is not None: m = [{'role': 'system', 'content': system}, m[1]]
        prompts.append(render(m, True))
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    outs = [None] * len(prompts); s = 0
    while s < len(order):
        e = s + 1
        while e < len(order) and e - s < bs and (e - s + 1) * len(prompts[order[e]]) <= char_budget: e += 1
        ids = order[s:e]
        enc = tok([prompts[i] for i in ids], return_tensors='pt', padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, do_sample=False, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **gen_kw)
        for j, i in enumerate(ids):
            outs[i] = tok.decode(gen[j, enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        s = e
    return outs

def eval_model(model, name, system=None, with_probe=True, gen_bs=16):
    t0 = time.time()
    out = generate(model, dev, system=system, bs=gen_bs)
    preds = [{'id': r['id'], 'output': o} for r, o in zip(dev, out)]
    write_jsonl(f'{RUNS}/{name}/dev_pred.jsonl', preds)
    s = score(dev, preds); s2 = score([r for r in dev if r['id'] not in LEAKED_DEV_IDS], preds)
    m = dict(dev_em=s['all']['exact_match'], dev_field=s['all']['mean_field_acc'], dev_json_valid=s['all']['json_valid'],
             dev_em_noleak=s2['all']['exact_match'], field_acc=s['all']['fields'])
    if with_probe:
        po = generate(model, probe, bs=gen_bs)
        ps = score(probe, [{'id': r['id'], 'output': o} for r, o in zip(probe, po)])
        m['probe_date_acc'] = ps['all']['fields']['txn_date']; m['probe_ok'] = True
    m['eval_min'] = (time.time() - t0) / 60
    return m

In [15]:
import shutil
def write_jsonl(path, rows):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + '\n')

RES_PATH = f'{RUNS}/results.json'
LOG_PATH = f'{WORK}/experiments_log.json'      # copy of results.json kept next to the notebook
if not os.path.exists(RES_PATH) and os.path.exists(LOG_PATH): shutil.copy(LOG_PATH, RES_PATH)
def load_results(): return json.load(open(RES_PATH)) if os.path.exists(RES_PATH) else {}
def save_result(name, d):
    res = load_results(); res[name] = d; json.dump(res, open(RES_PATH, 'w'), indent=1)
def write_results(res): json.dump(res, open(RES_PATH, 'w'), indent=1)

# Re-score every saved run from its saved dev predictions. This repairs any run scored by a wrong scorer.
res = load_results(); rows = []
for name, rec in res.items():
    p = f'{RUNS}/{name}/dev_pred.jsonl'
    if not os.path.exists(p): print('no saved dev predictions for', name); continue
    preds = read_jsonl(p)
    s = score(dev, preds)['all']; s2 = score([r for r in dev if r['id'] not in LEAKED_DEV_IDS], preds)['all']
    rec.update(dev_em=s['exact_match'], dev_field=s['mean_field_acc'], dev_json_valid=s['json_valid'],
               dev_em_noleak=s2['exact_match'], field_acc=s['fields'])
    rows.append((name, s))
write_results(res)
print(f"{'run':18}{'valid%':>7}{'EM%':>7}{'fields%':>8}")
for n, s in rows: print(f"{n:18}{100*s['json_valid']:7.1f}{100*s['exact_match']:7.1f}{100*s['mean_field_acc']:8.1f}")
if rows: print('\nper-field (last run above):', {k: round(v, 3) for k, v in rows[-1][1]['fields'].items()})

run                valid%    EM% fields%
base_fixed_prompt     0.0    0.0     0.0
base_rules_prompt   100.0    0.0    48.9
ref                 100.0   87.5    97.6
raw                 100.0   82.0    96.8
qlora               100.0   79.0    96.0
lr_1e-4             100.0   61.5    93.1
lr_5e-4             100.0   93.5    98.6
aug_dates           100.0   90.0    98.2
aug_lr5e-4          100.0   94.5    98.9

per-field (last run above): {'category': 0.975, 'priority': 0.965, 'amount_paise': 0.995, 'txn_date': 1.0, 'txn_id': 0.995, 'channel': 1.0, 'language': 1.0, 'needs_human': 0.98}


In [16]:
def ensure_probe(names, gen_bs=16):
    res = load_results()
    for n in names:
        rec = res.get(n)
        if not rec or not rec.get('adapter') or rec.get('probe_ok'): continue
        base = load_base(rec['cfg']['quant4']); mdl = PeftModel.from_pretrained(base, rec['adapter']); to_eval(mdl)
        po = generate(mdl, probe, bs=gen_bs)
        ps = score(probe, [{'id': r['id'], 'output': o} for r, o in zip(probe, po)])
        rec['probe_date_acc'] = ps['all']['fields']['txn_date']; rec['probe_ok'] = True; save_result(n, rec)
        print(n, 'probe txn_date accuracy: %.1f%%' % (100 * rec['probe_date_acc'])); del mdl, base; free()
_res = load_results()
for _r in _res.values():                     # probe values from runs scored before the scorer was fixed are not trustworthy
    if not _r.get('probe_ok'): _r.pop('probe_date_acc', None)
write_results(_res)
ensure_probe(['ref', 'aug_dates', 'lr_5e-4', 'aug_lr5e-4'])

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

lr_5e-4 probe txn_date accuracy: 58.1%


In [17]:
def exact_flags(name):
    preds = {p['id']: p['output'] for p in read_jsonl(f'{RUNS}/{name}/dev_pred.jsonl')}
    out = {}
    for r in dev:
        pr = parse(preds[r['id']]); g = json.loads(A(r))
        out[r['id']] = pr is not None and set(pr) == set(KEYS) and all(same(pr[k], g[k]) for k in KEYS)
    return out
ref_f = exact_flags('ref')
for n, rec in load_results().items():
    f = exact_flags(n)
    print(f"{n:18} right where ref is wrong: {sum(f[i] and not ref_f[i] for i in f):3} | wrong where ref is right: {sum(ref_f[i] and not f[i] for i in f):3} | final train loss: {rec.get('final_loss')}")
for n in ('base_fixed_prompt', 'base_rules_prompt'):
    print(n, '->', repr(read_jsonl(f'{RUNS}/{n}/dev_pred.jsonl')[0]['output'][:160]))

base_fixed_prompt  right where ref is wrong:   0 | wrong where ref is right: 175 | final train loss: None
base_rules_prompt  right where ref is wrong:   0 | wrong where ref is right: 175 | final train loss: None
ref                right where ref is wrong:   0 | wrong where ref is right:   0 | final train loss: 0.005475169050623663
raw                right where ref is wrong:   7 | wrong where ref is right:  18 | final train loss: 0.012957343156449497
qlora              right where ref is wrong:   3 | wrong where ref is right:  20 | final train loss: 0.006835874862736091
lr_1e-4            right where ref is wrong:   3 | wrong where ref is right:  55 | final train loss: 0.01349605533760041
lr_5e-4            right where ref is wrong:  17 | wrong where ref is right:   5 | final train loss: 0.003816675861889962
aug_dates          right where ref is wrong:  12 | wrong where ref is right:   7 | final train loss: 0.0014080146560445429
aug_lr5e-4         right where ref is wrong:  18 | wrong

In [18]:
RULES_PROMPT = '''You are NimbusPay's ticket triage model. Read the ticket and return ONLY a JSON object with exactly these 8 keys, in this order: category, priority, amount_paise, txn_date, txn_id, channel, language, needs_human. Use null when a value is not present.
category: fraud (customer says a transaction happened that they did not make, approve or recognise; wins over every other category) | payment_failed (a payment, transfer or top-up failed, was declined, is stuck, or did not reach the recipient) | refund (money owed back: refund asked or missing, cancelled order, return, charged more than once) | account_access (locked or blocked account, blocked PIN, forgotten passcode or password, OTP not received, reset problems) | kyc (identity verification, documents, Aadhaar, PAN, video KYC) | offers (cashback, promo codes, coupons, scratch cards, referral bonuses, reward points) | other (anything else).
priority: P1 if category is fraud, or the disputed amount is Rs 50,000 or more, or the customer makes a legal threat (RBI, ombudsman, consumer court, lawyer, legal notice, legal action). Otherwise P2 if category is payment_failed or refund with amount Rs 5,000 or more, or category is account_access. Otherwise P3.
amount_paise: integer paise (Rs 1 = 100 paise) of the transaction the ticket is about. Ignore balance, limits and fees. k means thousand, lakh means 100000. null if no amount.
txn_date: YYYY-MM-DD of the transaction. Resolve relative dates (yesterday, 3 days ago, kal, parso) against the Received date. If the year is missing use the most recent such date on or before Received. Ignore dates in quoted lines starting with >. null if not stated.
txn_id: NP followed by 10 digits, uppercase, no spaces or hyphens. Other reference numbers are not transaction ids. null if absent.
channel: upi, card, netbanking or wallet, only when explicitly stated, else null.
language: hi-en if the message is Hinglish, else en.
needs_human: true if priority is P1 or the customer says they already contacted support about this before (third time, again and again, baar baar), else false.
For account_access, kyc and other tickets, amount_paise, txn_date, txn_id and channel are always null.'''

def run_baseline(name, change, system):
    res = load_results()
    if name in res: print('skip (saved):', name); return res[name]
    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    model = load_base(False)
    m = eval_model(model, name, system=system, with_probe=False)
    rec = dict(name=name, change=change, cfg=None, adapter=None, peak_gb=torch.cuda.max_memory_allocated() / 2**30,
               train_min=0.0, **m)
    save_result(name, rec); del model; free(); return rec

b1 = run_baseline('base_fixed_prompt', 'Base model, fixed system prompt only', system=None)
b2 = run_baseline('base_rules_prompt', 'Base model, rules written in the prompt (reference only)', system=RULES_PROMPT)
for b in (b1, b2): print(b['name'], 'valid %.1f%%  EM %.1f%%  fields %.1f%%' % (100 * b['dev_json_valid'], 100 * b['dev_em'], 100 * b['dev_field']))

skip (saved): base_fixed_prompt
skip (saved): base_rules_prompt
base_fixed_prompt valid 0.0%  EM 0.0%  fields 0.0%
base_rules_prompt valid 100.0%  EM 0.0%  fields 48.9%


In [19]:
def make_windows(lengths, epochs, max_tokens, max_bs, target_ex, seed):
    """Plan every optimiser step up front. Shuffle, sort by length inside chunks of 256, pack micro-batches
    under a token budget (long threads run alone), then group micro-batches into steps of about target_ex examples."""
    windows = []
    for e in range(epochs):
        rng = random.Random(seed + e); idx = list(range(len(lengths))); rng.shuffle(idx)
        batches = []
        for c in range(0, len(idx), 256):
            chunk = sorted(idx[c:c + 256], key=lambda i: lengths[i]); cur = []; curmax = 0
            for i in chunk:
                m = max(curmax, lengths[i])
                if cur and (m * (len(cur) + 1) > max_tokens or len(cur) >= max_bs):
                    batches.append(cur); cur = []; m = lengths[i]
                cur.append(i); curmax = m
            if cur: batches.append(cur)
        rng.shuffle(batches)
        win, n = [], 0
        for b in batches:
            win.append(b); n += len(b)
            if n >= target_ex: windows.append(win); win, n = [], 0
        if win: windows.append(win)
    return windows

def collate(idxs, enc, device='cuda'):
    L = max(len(enc[i][0]) for i in idxs)
    ids = torch.full((len(idxs), L), tok.pad_token_id, dtype=torch.long)
    lab = torch.full((len(idxs), L), -100, dtype=torch.long)
    att = torch.zeros((len(idxs), L), dtype=torch.long)
    for k, i in enumerate(idxs):
        a, b = enc[i]
        ids[k, :len(a)] = torch.tensor(a); lab[k, :len(b)] = torch.tensor(b); att[k, :len(a)] = 1
    return ids.to(device), lab.to(device), att.to(device)

def train_run(name, rows, cfg):
    cfg = {**DEFAULT_CFG, **cfg}
    random.seed(cfg['seed']); np.random.seed(cfg['seed']); torch.manual_seed(cfg['seed']); torch.cuda.manual_seed_all(cfg['seed'])
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); t0 = time.time()

    model = load_base(cfg['quant4'])
    if cfg['quant4']:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                                gradient_checkpointing_kwargs={'use_reentrant': False})
    else:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        model.enable_input_require_grads()
    model.config.use_cache = False
    model = get_peft_model(model, LoraConfig(r=cfg['r'], lora_alpha=cfg['alpha'], lora_dropout=cfg['dropout'],
                                             target_modules=TARGETS, bias='none', task_type='CAUSAL_LM'))
    for p in model.parameters():
        if p.requires_grad: p.data = p.data.float()            # fp32 adapter weights, fp16 autocast compute
    params = [p for p in model.parameters() if p.requires_grad]
    n_train = sum(p.numel() for p in params)
    assert n_train < 50_000_000, f'{n_train} trainable parameters, over the 50M cap'
    print(f'[{name}] trainable parameters: {n_train / 1e6:.1f}M')

    enc = [encode_example(r) for r in rows]
    assert max(len(e[0]) for e in enc) <= MAX_LEN
    lengths = [len(e[0]) for e in enc]; nlab = [sum(1 for x in e[1] if x != -100) for e in enc]
    windows = make_windows(lengths, cfg['epochs'], cfg['max_tokens'], cfg['max_bs'], cfg['target_ex'], cfg['seed'])
    total = len(windows); warm = max(1, int(cfg['warmup'] * total))
    lr_at = lambda s: cfg['lr'] * ((s + 1) / warm if s < warm else 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))

    opt = torch.optim.AdamW(params, lr=cfg['lr'], weight_decay=0.0)
    try: scaler = torch.amp.GradScaler('cuda')
    except Exception: scaler = torch.cuda.amp.GradScaler()
    inner = model.get_base_model(); decoder = inner.model; lm_head = inner.get_output_embeddings()
    model.train(); curve = []
    for step, window in enumerate(windows):
        for g in opt.param_groups: g['lr'] = lr_at(step)
        ntok = sum(nlab[i] for mb in window for i in mb); step_loss = 0.0
        for mb in window:
            ids, lab, att = collate(mb, enc)
            with torch.autocast('cuda', dtype=torch.float16):
                h = decoder(input_ids=ids, attention_mask=att, use_cache=False).last_hidden_state[:, :-1]
                tgt = lab[:, 1:]; sel = tgt != -100
                logits = lm_head(h[sel])                       # head only at label positions
            loss = F.cross_entropy(logits.float(), tgt[sel], reduction='sum') / ntok
            scaler.scale(loss).backward(); step_loss += loss.item()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        curve.append(step_loss)
        if step % 20 == 0 or step == total - 1:
            print(f'[{name}] step {step + 1}/{total}  loss {step_loss:.4f}  lr {lr_at(step):.2e}')
    info = dict(peak_gb=torch.cuda.max_memory_allocated() / 2**30, train_min=(time.time() - t0) / 60,
                n_trainable=n_train, steps=total, n_rows=len(rows), final_loss=float(np.mean(curve[-10:])))
    print(f'[{name}] peak GPU {info["peak_gb"]:.1f} GB, {info["train_min"]:.1f} min')
    return model, info

In [20]:
_len = [int(x) for x in lens[:600]]
w = make_windows(_len, epochs=2, max_tokens=8192, max_bs=16, target_ex=16, seed=0)
seen = collections.Counter(i for win in w for mb in win for i in mb)
assert all(v == 2 for v in seen.values()) and len(seen) == len(_len), 'every example exactly once per epoch'
assert all(max(_len[i] for i in mb) * len(mb) <= 8192 or len(mb) == 1 for win in w for mb in win), 'token budget respected'
print('steps:', len(w), '| largest micro-batch (examples):', max(len(mb) for win in w for mb in win))

steps: 72 | largest micro-batch (examples): 16


In [21]:
def run_experiment(name, change, rows, cfg, force=False):
    res = load_results()
    if name in res and not force: print('skip (saved):', name); return res[name]
    os.makedirs(f'{RUNS}/{name}', exist_ok=True)
    model, info = train_run(name, rows, cfg)
    adapter_dir = f'{RUNS}/{name}/adapter'; model.save_pretrained(adapter_dir)
    to_eval(model); m = eval_model(model, name)
    rec = dict(name=name, change=change, cfg={**DEFAULT_CFG, **cfg}, adapter=adapter_dir, **info, **m)
    save_result(name, rec); del model; free()
    print(f'>>> {name}: EM {100 * rec["dev_em"]:.1f}%  fields {100 * rec["dev_field"]:.1f}%  probe date {100 * rec["probe_date_acc"]:.1f}%')
    return rec

EXPERIMENTS = {
    'ref':        ('Reference: cleaned data, 16-bit LoRA, r16, lr 2e-4, 3 epochs', train_clean,             {}),
    'raw':        ('Raw train.jsonl instead of cleaned (all 1,445 rows)',            train_raw,               {}),
    'qlora':      ('4-bit NF4 QLoRA instead of 16-bit',                             train_clean,             dict(quant4=True)),
    'lr_1e-4':    ('Learning rate 1e-4',                                            train_clean,             dict(lr=1e-4)),
    'lr_5e-4':    ('Learning rate 5e-4',                                            train_clean,             dict(lr=5e-4)),
    'aug_dates':  ('Cleaned data + date-shift augmentation',                        train_clean + aug_rows,  {}),
    'aug_lr5e-4': ('Combination: augmentation + lr 5e-4 (the two best single changes)', train_clean + aug_rows, dict(lr=5e-4)),
}
RUN_NOW = list(EXPERIMENTS)      # finished runs are skipped, so this is instant when everything is saved
for n in RUN_NOW: run_experiment(n, *EXPERIMENTS[n])

skip (saved): ref
skip (saved): raw
skip (saved): qlora
skip (saved): lr_1e-4
skip (saved): lr_5e-4
skip (saved): aug_dates
skip (saved): aug_lr5e-4


In [22]:
import pandas as pd
def results_table():
    rows = []
    for r in load_results().values():
        rows.append({'run': r['name'], 'what changed': r['change'], 'dev EM %': round(100 * r['dev_em'], 1),
                     'dev field acc %': round(100 * r['dev_field'], 1), 'EM excl. leaked %': round(100 * r['dev_em_noleak'], 1),
                     'probe date %': round(100 * r['probe_date_acc'], 1) if 'probe_date_acc' in r else None,
                     'peak GB': round(r['peak_gb'], 1), 'train min': round(r['train_min'], 1)})
    return pd.DataFrame(rows)
tbl = results_table(); display(tbl)
try: print(tbl.to_markdown(index=False))
except ImportError: print(tbl.to_string(index=False))
shutil.copy(RES_PATH, LOG_PATH)    # keep a copy next to the notebook so the submitted version can show earlier runs

,run,what changed,dev EM %,dev field acc %,EM excl. leaked %,probe date %,peak GB,train min
0,base_fixed_prompt,"Base model, fixed system prompt only",0.0,0.0,0.0,NaN,6.0,0.0
1,base_rules_prompt,"Base model, rules written in the prompt (refer...",0.0,48.9,0.0,NaN,6.0,0.0
2,ref,"Reference: cleaned data, 16-bit LoRA, r16, lr ...",87.5,97.6,87.6,57.0,8.9,13.7
3,raw,"Raw train.jsonl instead of cleaned (all 1,445 ...",82.0,96.8,80.0,NaN,8.9,14.5
4,qlora,4-bit NF4 QLoRA instead of 16-bit,79.0,96.0,78.8,NaN,8.3,16.7
5,lr_1e-4,Learning rate 1e-4,61.5,93.1,62.4,NaN,8.9,13.7
6,lr_5e-4,Learning rate 5e-4,93.5,98.6,92.9,58.1,8.9,13.6
7,aug_dates,Cleaned data + date-shift augmentation,90.0,98.2,90.0,94.2,8.9,16.1
8,aug_lr5e-4,Combination: augmentation + lr 5e-4 (the two b...,94.5,98.9,94.1,98.8,8.9,15.4


| run               | what changed                                                      |   dev EM % |   dev field acc % |   EM excl. leaked % |   probe date % |   peak GB |   train min |
|:------------------|:------------------------------------------------------------------|-----------:|------------------:|--------------------:|---------------:|----------:|------------:|
| base_fixed_prompt | Base model, fixed system prompt only                              |        0   |               0   |                 0   |          nan   |       6   |         0   |
| base_rules_prompt | Base model, rules written in the prompt (reference only)          |        0   |              48.9 |                 0   |          nan   |       6   |         0   |
| ref               | Reference: cleaned data, 16-bit LoRA, r16, lr 2e-4, 3 epochs      |       87.5 |              97.6 |                87.6 |           57   |       8.9 |        13.7 |
| raw               | Raw train.jsonl instead of cleaned (al

'/content/drive/MyDrive/vima3ya/experiments_log.json'

In [23]:
FINAL_RUN = 'aug_lr5e-4'   # chosen by a rule fixed in advance (at least 92% dev EM), also the best run by dev EM
trained = [r for r in load_results().values() if r.get('adapter')]
best = load_results()[FINAL_RUN] if FINAL_RUN else max(trained, key=lambda r: (r['dev_em'], r['dev_field']))
if not os.path.isdir(best['adapter']):          # only the log survived, retrain this one run
    best = run_experiment(best['name'], *EXPERIMENTS[best['name']], force=True)
print('best run:', best['name'], '| dev EM %.1f%% | fields %.1f%%' % (100 * best['dev_em'], 100 * best['dev_field']))

ADAPTER_DIR = f'{OUT}/adapter'
shutil.copytree(best['adapter'], ADAPTER_DIR, dirs_exist_ok=True)
print(sorted(os.listdir(ADAPTER_DIR)))
cfg_json = json.load(open(f'{ADAPTER_DIR}/adapter_config.json'))
print({k: cfg_json[k] for k in ('base_model_name_or_path', 'r', 'lora_alpha', 'target_modules', 'peft_type')})
from safetensors import safe_open
with safe_open(f'{ADAPTER_DIR}/adapter_model.safetensors', 'pt') as f:
    n_params = sum(f.get_tensor(k).numel() for k in f.keys())
print(f'trainable parameters stored in adapter: {n_params / 1e6:.1f}M (cap 50M)'); assert n_params < 50_000_000
print('load base in 4-bit?', best['cfg']['quant4'])

best run: aug_lr5e-4 | dev EM 94.5% | fields 98.9%
['README.md', 'adapter_config.json', 'adapter_model.safetensors']
{'base_model_name_or_path': 'Qwen/Qwen2.5-1.5B-Instruct', 'r': 16, 'lora_alpha': 32, 'target_modules': ['k_proj', 'v_proj', 'o_proj', 'q_proj', 'gate_proj', 'down_proj', 'up_proj'], 'peft_type': 'LORA'}
trainable parameters stored in adapter: 18.5M (cap 50M)
load base in 4-bit? False


In [24]:
ADAPTER_DIR = f'{OUT}/adapter'
base = load_base(best['cfg']['quant4'])
final = PeftModel.from_pretrained(base, ADAPTER_DIR); to_eval(final)
print('generation_config defaults -> repetition_penalty:', getattr(final.generation_config, 'repetition_penalty', None), '| do_sample:', final.generation_config.do_sample)

sub = dev[:60]
for label, kw in (('default call', {}), ('repetition_penalty=1.0', dict(repetition_penalty=1.0))):
    o = generate(final, sub, bs=8, **kw); s_ = score(sub, [{'id': r['id'], 'output': x} for r, x in zip(sub, o)])['all']
    print(f"{label:24} valid {100*s_['json_valid']:.1f}%  EM {100*s_['exact_match']:.1f}%  fields {100*s_['mean_field_acc']:.1f}%")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config defaults -> repetition_penalty: 1.1 | do_sample: True
default call             valid 100.0%  EM 88.3%  fields 97.3%
repetition_penalty=1.0   valid 100.0%  EM 90.0%  fields 97.7%


In [25]:
TEST_BS = 1          # final submission pass, exactly how reviewers run it (about 35 min, resumable). 8 would be about 6 min.
FINAL_GEN_KW = {}    # {} = plain greedy call. Only change this if cell 24 showed a clear difference, and then say so in the README.
PART = f'{OUT}/test_partial_{best["name"]}_bs{TEST_BS}.json'      # resumable: a disconnect only loses the current chunk
done = json.load(open(PART)) if os.path.exists(PART) else {}
todo = [r for r in test_in if r['id'] not in done]
t0 = time.time()
for i in range(0, len(todo), 20):
    chunk = todo[i:i + 20]
    outs = generate(final, chunk, bs=TEST_BS, **FINAL_GEN_KW)
    done.update({r['id']: o for r, o in zip(chunk, outs)}); json.dump(done, open(PART, 'w'))
    print(f'{len(done)}/{len(test_in)}  {(time.time() - t0) / 60:.1f} min', end='\r')
test_pred = [{'id': r['id'], 'output': done[r['id']]} for r in test_in]
write_jsonl(f'{OUT}/predictions_test.jsonl', test_pred)
print(f'\nwrote {len(test_pred)} predictions, {sum(parse(p["output"]) is not None for p in test_pred)} parse as JSON objects')

sample = random.Random(1).sample(range(len(test_in)), 20)
again = generate(final, [test_in[i] for i in sample], bs=1, **FINAL_GEN_KW)
print('identical to the saved predictions:', sum(again[k] == test_pred[i]['output'] for k, i in enumerate(sample)), '/ 20')


wrote 400 predictions, 400 parse as JSON objects
identical to the saved predictions: 20 / 20


In [26]:
a_p = f'{OUT}/test_partial_{best["name"]}_bs8.json'; b_p = f'{OUT}/test_partial_{best["name"]}_bs1.json'
if os.path.exists(a_p) and os.path.exists(b_p):
    a = json.load(open(a_p)); b = json.load(open(b_p))
    print(sum(a[k] != b[k] for k in a), 'of', len(a), 'test rows differ between bs=8 and bs=1')
else:
    print('no bs=8 file to compare with (fine, it only existed from the first pass)')

0 of 400 test rows differ between bs=8 and bs=1


In [27]:
dev_pred = read_jsonl(f'{RUNS}/{best["name"]}/dev_pred.jsonl')
res = score(dev, dev_pred)
print('per-field accuracy'); [print(f'  {k:14} {v:6.1%}') for k, v in res['all']['fields'].items()]
cat_rows = [{'category': n.split('=')[1], 'n': r['n'], 'exact %': round(100 * r['exact_match'], 1), 'field acc %': round(100 * r['mean_field_acc'], 1),
             **{k: round(100 * v, 1) for k, v in r['fields'].items() if v < 1}} for n, r in res.items() if n.startswith('category=')]
display(pd.DataFrame(cat_rows).sort_values('exact %'))

long_ids = {r['id'] for r in dev if len(U(r)) > 1500}
for nm, sel in (('long quoted threads', lambda r: r['id'] in long_ids), ('regular tickets', lambda r: r['id'] not in long_ids)):
    g = [r for r in dev if sel(r)]; ss = score(g, dev_pred)['all']
    print(f'{nm:22} n={ss["n"]:3}{"  (tiny sample, interpret with care)" if ss["n"] < 20 else ""}  exact {ss["exact_match"]:.1%}  fields {ss["mean_field_acc"]:.1%}')

per-field accuracy
  category        97.5%
  priority        96.5%
  amount_paise    99.5%
  txn_date       100.0%
  txn_id          99.5%
  channel        100.0%
  language       100.0%
  needs_human     98.0%


,category,n,exact %,field acc %,priority,amount_paise,needs_human,txn_id
4,86.7,15,86.7,96.7,86.7,NaN,NaN,NaN
3,96.2,53,90.6,98.1,94.3,NaN,96.2,98.1
0,93.8,16,93.8,99.2,NaN,NaN,NaN,NaN
2,refund,55,94.5,98.9,96.4,98.2,96.4,NaN
1,kyc,24,100.0,100.0,NaN,NaN,NaN,NaN
5,offers,18,100.0,100.0,NaN,NaN,NaN,NaN
6,account_access,19,100.0,100.0,NaN,NaN,NaN,NaN


long quoted threads    n=  3  (tiny sample, interpret with care)  exact 100.0%  fields 100.0%
regular tickets        n=197  exact 94.4%  fields 98.9%


In [28]:
fails = []
for r, p in zip(dev, dev_pred):
    gold = json.loads(A(r)); pr = parse(p['output'])
    wrong = [k for k in KEYS if pr is None or k not in pr or not same(pr[k], gold[k])]
    if wrong or pr is None or set(pr) != set(KEYS): fails.append((r, gold, pr, p['output'], wrong))
print(len(fails), 'dev rows with at least one wrong field')
by_field = collections.Counter(k for *_, w in fails for k in w); print(dict(by_field))

shown, per_field = [], collections.Counter()
for f in fails:
    for k in f[4]:
        if per_field[k] < 3 and f not in shown: shown.append(f); per_field[k] += 1
for r, gold, pr, raw, wrong in shown[:18]:
    print('-' * 100); print(r['id'], '| wrong:', wrong)
    print('TEXT:', re.sub(r'\s+', ' ', U(r).split('\n\n', 1)[-1])[:300])
    for k in wrong: print(f'   {k:13} gold={gold[k]!r:22} pred={(pr or {}).get(k, "<invalid json>")!r}')

11 dev rows with at least one wrong field
{'priority': 7, 'needs_human': 4, 'category': 5, 'amount_paise': 1, 'txn_id': 1}
----------------------------------------------------------------------------------------------------
dv-00005 | wrong: ['priority', 'needs_human']
TEXT: [IVR transcript] Hey, I tried to make a transfer but it failed. Dated Aug 1. Total 49680 rupees. Ref: NP9512428472 Kindly resolve this asap.
   priority      gold='P2'                   pred='P1'
   needs_human   gold=False                  pred=True
----------------------------------------------------------------------------------------------------
dv-00010 | wrong: ['category', 'priority', 'needs_human']
TEXT: Subject: Problem with NimbusPay URGENT!! I didn't see the money I added to my card. ID: NP0194178721 INR 2,570 was the value. Please look into it. Regards, Vikram Singh
   category      gold='payment_failed'       pred='fraud'
   priority      gold='P3'                   pred='P1'
   needs_human   gold=Fals

In [29]:
for r, gold, pr, raw, wrong in fails:
    print('-' * 100); print(r['id'], '| wrong:', wrong)
    print('TEXT:', re.sub(r'\s+', ' ', U(r).split('\n\n', 1)[-1])[:300])
    for k in wrong: print(f'   {k:13} gold={gold[k]!r:22} pred={(pr or {}).get(k, "<invalid json>")!r}')
print(pd.DataFrame(cat_rows).sort_values('exact %').to_string(index=False))

----------------------------------------------------------------------------------------------------
dv-00005 | wrong: ['priority', 'needs_human']
TEXT: [IVR transcript] Hey, I tried to make a transfer but it failed. Dated Aug 1. Total 49680 rupees. Ref: NP9512428472 Kindly resolve this asap.
   priority      gold='P2'                   pred='P1'
   needs_human   gold=False                  pred=True
----------------------------------------------------------------------------------------------------
dv-00010 | wrong: ['category', 'priority', 'needs_human']
TEXT: Subject: Problem with NimbusPay URGENT!! I didn't see the money I added to my card. ID: NP0194178721 INR 2,570 was the value. Please look into it. Regards, Vikram Singh
   category      gold='payment_failed'       pred='fraud'
   priority      gold='P3'                   pred='P1'
   needs_human   gold=False                  pred=True
----------------------------------------------------------------------------------------------

In [30]:
ids = ['dv-00017', 'dv-00145', 'dv-00005', 'dv-00037', 'dv-00056', 'dv-00010', 'dv-00042', 'dv-00046']
rows = [r for r in dev if r['id'] in ids]
for label, kw in (('default', {}), ('repetition_penalty=1.0', dict(repetition_penalty=1.0))):
    o = generate(final, rows, bs=1, **kw)
    ok = [parse(x) is not None and all(same(parse(x)[k], json.loads(A(r))[k]) for k in KEYS) for r, x in zip(rows, o)]
    print(f'{label:24} exact: {sum(ok)}/{len(rows)}  fixed rows: {[r["id"] for r, k in zip(rows, ok) if k]}')

default                  exact: 0/8  fixed rows: []
repetition_penalty=1.0   exact: 1/8  fixed rows: ['dv-00056']


In [31]:
from google.colab import files
SUB = '/content/submission'; shutil.rmtree(SUB, ignore_errors=True); os.makedirs(f'{SUB}/adapter')
for f in ('adapter_config.json', 'adapter_model.safetensors'): shutil.copy(f'{OUT}/adapter/{f}', f'{SUB}/adapter/{f}')
shutil.copy(f'{OUT}/cleaning_log.csv', SUB); shutil.copy(f'{OUT}/predictions_test.jsonl', SUB); shutil.copy(LOG_PATH, f'{SUB}/experiments_log.json')

pred = read_jsonl(f'{SUB}/predictions_test.jsonl')
assert [p['id'] for p in pred] == [r['id'] for r in test_in], 'ids or order differ from test_inputs.jsonl'
assert all(set(p) == {'id', 'output'} for p in pred)
ok = sum(1 for p in pred if parse(p['output']) is not None and set(parse(p['output'])) == set(KEYS))
print(f'predictions: {len(pred)} rows | valid JSON with exactly the 8 keys: {ok}')
print('cleaning_log rows:', len(list(csv.DictReader(open(f'{SUB}/cleaning_log.csv')))))
print('adapter size: %.0f MB (GitHub limit 100 MB, so no LFS needed if under)' % (os.path.getsize(f'{SUB}/adapter/adapter_model.safetensors') / 2**20))
print('adapter base model:', json.load(open(f'{SUB}/adapter/adapter_config.json'))['base_model_name_or_path'])
shutil.make_archive('/content/submission_files', 'zip', SUB); files.download('/content/submission_files.zip')

predictions: 400 rows | valid JSON with exactly the 8 keys: 400
cleaning_log rows: 300
adapter size: 70 MB (GitHub limit 100 MB, so no LFS needed if under)
adapter base model: Qwen/Qwen2.5-1.5B-Instruct


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>